# SMART Baseline Visualization (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-baseline/notebooks/smart_baseline_visualize_colab.ipynb)

This notebook owns the **qualitative visualization stage**:
- load a checkpointed SMART baseline rollout,
- choose a representative scenario from official metrics,
- reconstruct token-step decisions for one focal agent,
- render a closed-loop rollout MP4,
- save the video and metadata to Drive.


In [1]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config

runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] Starting deterministic environment bootstrap
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, jax.__version__)
[setup] Core runtime already healthy; skipping dependency install (ok 2.2.6 2.2.3 1.14.1 1.6.1 0.7.2).
[setup] $ /usr/bin/python3 -c import numpy as np; from numpy._core.umath import _center, _expandtabs; print(np.__version__, np.__file__)
[setup] NumPy probe passed (2.2.6 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py).
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__

In [2]:
# Suggested defaults: visualize the latest SMART baseline eval run and save the MP4 to Drive.
%env WOSAC_RUN_MODE=full
%env WOSAC_AUTO_RESUME=1
%env SMART_PROCESSED_DATA_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
%env SMART_VAL_CONFIG=experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml
%env WOSAC_SCENARIO_TFRECORDS=gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-*
%env SMART_VIS_SELECTION_STRATEGY=representative_safe
%env SMART_VIS_SCENARIO_ID=
%env SMART_VIS_ROLLOUT_INDEX=0
%env SMART_VIS_FOCAL_OBJECT_ID=0
%env SMART_VIS_FPS=10
%env SMART_VIS_DPI=120
%env SMART_VIS_VIEW_RADIUS=0
%env SMART_VIS_MAX_CANDIDATES=5


env: WOSAC_RUN_MODE=full
env: WOSAC_AUTO_RESUME=1
env: SMART_PROCESSED_DATA_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
env: SMART_VAL_CONFIG=experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml
env: WOSAC_SCENARIO_TFRECORDS=gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-*
env: SMART_VIS_SELECTION_STRATEGY=representative_safe
env: SMART_VIS_SCENARIO_ID=
env: SMART_VIS_ROLLOUT_INDEX=0
env: SMART_VIS_FOCAL_OBJECT_ID=0
env: SMART_VIS_FPS=10
env: SMART_VIS_DPI=120
env: SMART_VIS_VIEW_RADIUS=0
env: SMART_VIS_MAX_CANDIDATES=5


In [3]:
# Step 2: Resolve the baseline eval artifacts, checkpoint, and visualization output paths.
import json
import os
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config


def _bool_env(name: str, default: bool) -> bool:
    text = os.environ.get(name, '').strip().lower()
    if not text:
        return bool(default)
    return text in {'1', 'true', 'yes', 'y', 'on'}


def _latest_dir(root: Path, pattern: str) -> Path | None:
    if not root.exists():
        return None
    paths = [p for p in root.glob(pattern) if p.exists() and p.is_dir()]
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]


def _latest_ckpt(search_root: Path) -> str:
    if (not search_root.exists()) or (not search_root.is_dir()):
        return ''
    ckpts = [p for p in search_root.rglob('*.ckpt') if p.is_file()]
    if not ckpts:
        return ''
    ckpts.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return str(ckpts[0])


EXPERIMENT_SLUG = 'smart-baseline'
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root='.')
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root='.')
RUN = cfg.get('run', {})
SMART = cfg.get('smart', {})
SMART_PROFILES = SMART.get('profiles', {})
SMART_PROFILE = str(os.environ.get('SMART_BASELINE_PROFILE', SMART.get('profile', 'paper_repro'))).strip() or 'paper_repro'
SMART_EFFECTIVE = dict(SMART)
SMART_EFFECTIVE.update(SMART_PROFILES.get(SMART_PROFILE, {}))

PERSIST_ROOT = Path(str(os.environ.get('WOSAC_PERSIST_ROOT', RUN.get('persist_root', '/content/drive/MyDrive/wosac_experiments')))).expanduser()
RUN_PREFIX = str(RUN.get('run_prefix', 'smart_baseline'))
RUN_NAME = str(RUN.get('run_name', 'dev'))
OUTPUTS_DIR = PERSIST_ROOT / f'{RUN_PREFIX}_{RUN_NAME}' / 'outputs'
RUN_TAG = str(os.environ.get('WOSAC_RUN_TAG', '')).strip()
RUN_DIR = OUTPUTS_DIR / RUN_TAG if RUN_TAG else _latest_dir(OUTPUTS_DIR, '20*Z')
if RUN_DIR is None or (not RUN_DIR.exists()):
    raise RuntimeError(f'Could not resolve a baseline eval run directory under {OUTPUTS_DIR}')
RUN_TAG = RUN_DIR.name

SMART_PROCESSED_DATA_ROOT = str(os.environ.get('SMART_PROCESSED_DATA_ROOT', SMART.get('processed_data_root', '/content/SMART/data/waymo_processed'))).strip()
SMART_VAL_CONFIG = str(os.environ.get('SMART_VAL_CONFIG', SMART_EFFECTIVE.get('val_config', 'experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml'))).strip()
SCENARIO_PROTO_PATH = str(os.environ.get('WOSAC_SCENARIO_PROTO_PATH', '')).strip()
SCENARIO_PROTO_DIR = str(os.environ.get('WOSAC_SCENARIO_PROTO_DIR', '')).strip()
SCENARIO_TFRECORDS = str(os.environ.get('WOSAC_SCENARIO_TFRECORDS', '')).strip()
SELECTION_STRATEGY = str(os.environ.get('SMART_VIS_SELECTION_STRATEGY', 'representative_safe')).strip()
EXPLICIT_SCENARIO_ID = str(os.environ.get('SMART_VIS_SCENARIO_ID', '')).strip()
ROLLOUT_INDEX = int(os.environ.get('SMART_VIS_ROLLOUT_INDEX', '0'))
FOCAL_OBJECT_ID = int(os.environ.get('SMART_VIS_FOCAL_OBJECT_ID', '0'))
FPS = int(os.environ.get('SMART_VIS_FPS', '10'))
DPI = int(os.environ.get('SMART_VIS_DPI', '120'))
VIEW_RADIUS = float(os.environ.get('SMART_VIS_VIEW_RADIUS', '0'))
MAX_CANDIDATES = int(os.environ.get('SMART_VIS_MAX_CANDIDATES', '5'))
SKIP_TOKEN_TRACE = _bool_env('SMART_VIS_SKIP_TOKEN_TRACE', False)
SMART_REPO_URL = str(SMART_EFFECTIVE.get('repo_url', 'https://github.com/rainmaker22/SMART.git')).strip()
SMART_REPO_BRANCH = str(SMART_EFFECTIVE.get('branch', 'main')).strip() or 'main'
SMART_REPO_DIR = str(SMART_EFFECTIVE.get('repo_dir', '/content/SMART'))
SMART_BASELINE_CKPT = str(os.environ.get('SMART_BASELINE_CKPT', '')).strip()

rollout_proto_path = Path(str(os.environ.get('SMART_BASELINE_ROLLOUT_PROTO', RUN_DIR / 'rollout_protos' / 'smart_baseline.binproto'))).expanduser()
official_metrics_json = Path(str(os.environ.get('SMART_BASELINE_OFFICIAL_METRICS_JSON', RUN_DIR / 'official_metrics' / 'smart_baseline.json'))).expanduser()
visual_dir = Path(str(os.environ.get('SMART_VIS_OUTPUT_DIR', RUN_DIR / 'visualizations'))).expanduser()
visual_dir.mkdir(parents=True, exist_ok=True)

if not SMART_BASELINE_CKPT:
    smoke_root = PERSIST_ROOT / f'{RUN_PREFIX}_{RUN_NAME}' / 'checkpoints' / 'smart_baseline_smoke'
    paper_root = PERSIST_ROOT / f'{RUN_PREFIX}_{RUN_NAME}' / 'checkpoints' / 'smart_baseline_paper_repro'
    SMART_BASELINE_CKPT = _latest_ckpt(smoke_root) or _latest_ckpt(paper_root)
if not SMART_BASELINE_CKPT:
    raise RuntimeError('Could not resolve SMART_BASELINE_CKPT. Set SMART_BASELINE_CKPT explicitly.')

summary = {
    'pack_dir': str(bundle.paths.get('pack_dir', '')),
    'run_tag': RUN_TAG,
    'run_dir': str(RUN_DIR),
    'rollout_proto_path': str(rollout_proto_path),
    'official_metrics_json': str(official_metrics_json),
    'smart_baseline_ckpt': str(SMART_BASELINE_CKPT),
    'smart_processed_data_root': SMART_PROCESSED_DATA_ROOT,
    'smart_val_config': SMART_VAL_CONFIG,
    'smart_repo_dir': SMART_REPO_DIR,
    'scenario_tfrecords': SCENARIO_TFRECORDS,
    'selection_strategy': SELECTION_STRATEGY,
    'explicit_scenario_id': EXPLICIT_SCENARIO_ID,
    'rollout_index': ROLLOUT_INDEX,
    'focal_object_id': FOCAL_OBJECT_ID,
    'fps': FPS,
    'dpi': DPI,
    'view_radius': VIEW_RADIUS,
    'skip_token_trace': SKIP_TOKEN_TRACE,
    'visual_dir': str(visual_dir),
}
print(json.dumps(summary, indent=2, sort_keys=True))

if not rollout_proto_path.exists():
    raise FileNotFoundError(f'Missing rollout proto: {rollout_proto_path}')
if not official_metrics_json.exists() and not EXPLICIT_SCENARIO_ID:
    raise FileNotFoundError(
        f'Missing official metrics JSON required for automatic scenario selection: {official_metrics_json}'
    )


{
  "dpi": 120,
  "explicit_scenario_id": "",
  "focal_object_id": 0,
  "fps": 10,
  "official_metrics_json": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/official_metrics/smart_baseline.json",
  "pack_dir": "/content/wosac-sim-agents-experiments/experiments/smart-baseline",
  "rollout_index": 0,
  "rollout_proto_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/rollout_protos/smart_baseline.binproto",
  "run_dir": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z",
  "run_tag": "20260404T162859Z",
  "scenario_tfrecords": "gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-*",
  "selection_strategy": "representative_safe",
  "skip_token_trace": false,
  "smart_baseline_ckpt": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_smoke/epoch=30.ckpt",
  "smart_processed_data_root": "/content/dri

In [4]:
# Step 3: Rank candidate scenarios from official metrics and choose one for the video.
import json

from src.workflows import load_visualization_metrics, rank_visualization_candidates, select_visualization_scenario

metrics_payload = load_visualization_metrics(official_metrics_json) if official_metrics_json.exists() else {'per_scenario': []}
selected = select_visualization_scenario(
    metrics_payload,
    scenario_id=EXPLICIT_SCENARIO_ID,
    strategy=SELECTION_STRATEGY,
) if (EXPLICIT_SCENARIO_ID or metrics_payload.get('per_scenario')) else {'scenario_id': EXPLICIT_SCENARIO_ID}

candidates = rank_visualization_candidates(
    metrics_payload,
    strategy=SELECTION_STRATEGY,
    limit=MAX_CANDIDATES,
) if metrics_payload.get('per_scenario') else []

SELECTED_SCENARIO_ID = str(selected.get('scenario_id', '')).strip()
if not SELECTED_SCENARIO_ID:
    raise RuntimeError('Scenario selection failed. Set SMART_VIS_SCENARIO_ID explicitly.')

VIDEO_PATH = visual_dir / f'smart_baseline_{SELECTED_SCENARIO_ID}_rollout{ROLLOUT_INDEX}.mp4'
SUMMARY_JSON_PATH = VIDEO_PATH.with_suffix('.json')
TOKEN_TRACE_JSON_PATH = VIDEO_PATH.with_suffix('.token_trace.json')

selection_summary = {
    'selected': selected,
    'candidate_count': len(candidates),
    'video_path': str(VIDEO_PATH),
    'summary_json_path': str(SUMMARY_JSON_PATH),
    'token_trace_json_path': str(TOKEN_TRACE_JSON_PATH),
}
print(json.dumps(selection_summary, indent=2, sort_keys=True))

try:
    import pandas as pd
    display(pd.DataFrame(candidates))
except Exception:
    print('candidate_rows:')
    print(json.dumps(candidates, indent=2, sort_keys=True))


{
  "candidate_count": 1,
  "selected": {
    "metametric": 0.6964772939682007,
    "min_average_displacement_error": 8.540173530578613,
    "representative_gap": 0.0,
    "safe": true,
    "scenario_id": "9e455a0332beef76",
    "selection_pool": "safe",
    "selection_reason": "safe scenario nearest median metametric",
    "selection_strategy": "representative_safe",
    "simulated_collision_rate": 0.0,
    "simulated_offroad_rate": 0.0,
    "simulated_traffic_light_violation_rate": 0.0
  },
  "summary_json_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/visualizations/smart_baseline_9e455a0332beef76_rollout0.json",
  "token_trace_json_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/visualizations/smart_baseline_9e455a0332beef76_rollout0.token_trace.json",
  "video_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/visualizations/smart_baseline_9e455a0332

,scenario_id,metametric,min_average_displacement_error,simulated_collision_rate,simulated_offroad_rate,simulated_traffic_light_violation_rate,safe,selection_pool,selection_strategy,representative_gap
0,9e455a0332beef76,0.696477,8.540174,0.0,0.0,0.0,True,safe,representative_safe,0.0


In [5]:
# Step 4: Optional SMART setup + MP4 rendering.
import importlib
import json
import shlex
import subprocess
import sys
from pathlib import Path

from src.workflows.smart_eval_flow import _sync_git_repo

RUN_SETUP = _bool_env('SMART_RUN_SETUP', True)
RUN_RENDER = _bool_env('SMART_RUN_RENDER', True)
OVERWRITE_VIDEO = _bool_env('SMART_VIS_OVERWRITE', False)
video_exists = VIDEO_PATH.exists() and VIDEO_PATH.stat().st_size > 0

print('RUN_SETUP:', RUN_SETUP)
print('RUN_RENDER:', RUN_RENDER)
print('video_exists:', video_exists)
print('video_path:', VIDEO_PATH)

smart_repo_sync = _sync_git_repo(
    repo_url=SMART_REPO_URL,
    repo_branch=SMART_REPO_BRANCH,
    repo_dir=Path(SMART_REPO_DIR).expanduser(),
)
print('smart_repo_sync:')
print(json.dumps(smart_repo_sync, indent=2, sort_keys=True))


def _ensure_colab_gcs_auth_if_needed(paths: list[str], probe_glob: str = '') -> None:
    if 'google.colab' not in sys.modules:
        return
    if not any(str(p).strip().startswith('gs://') for p in paths if str(p).strip()):
        return
    from google.colab import auth  # type: ignore

    auth.authenticate_user()
    print('Colab auth: OK')
    try:
        active = subprocess.check_output(
            ['bash', '-lc', "gcloud auth list --filter=status:ACTIVE --format='value(account)'"],
            text=True,
        ).strip()
        print('Active gcloud account:', active or '<none>')
    except Exception as exc:
        print(f'Active gcloud account lookup failed: {exc}')

    if probe_glob.strip():
        try:
            tf = importlib.import_module('tensorflow')
            matches = list(tf.io.gfile.glob(probe_glob.strip()))[:3]
            print('GCS probe matches:', matches)
        except Exception as exc:
            print(f'GCS probe failed: {type(exc).__name__}: {exc}')
            raise


def _run_cmd(cmd: str, idx: int, total: int, *, label: str) -> None:
    print(f'[{label} {idx}/{total}] {cmd}')
    merged_cmd = f'export PYTHONUNBUFFERED=1; {cmd}'
    process = subprocess.Popen(
        ['bash', '-lc', merged_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        recent_lines.append(line)
        if len(recent_lines) > 100:
            recent_lines.pop(0)
    return_code = int(process.wait())
    if return_code == 0:
        return
    diag = {
        'failed_command': cmd,
        'exit_code': return_code,
        'video_path': str(VIDEO_PATH),
        'summary_json_path': str(SUMMARY_JSON_PATH),
        'recent_output': ''.join(recent_lines[-30:]),
    }
    print('step4_diagnostics:')
    print(json.dumps(diag, indent=2, sort_keys=True))
    raise subprocess.CalledProcessError(return_code, ['bash', '-lc', merged_cmd])


_ensure_colab_gcs_auth_if_needed(
    [str(SCENARIO_TFRECORDS), str(SCENARIO_PROTO_PATH), str(SCENARIO_PROTO_DIR)],
    probe_glob=str(SCENARIO_TFRECORDS),
)


if RUN_SETUP:
    setup_cmd = (
        f'cd {REPO_DIR} && python {REPO_DIR}/scripts/ensure_smart_train_runtime.py '
        f'--smart-repo-dir {SMART_REPO_DIR}'
    )
    _run_cmd(setup_cmd, 1, 1 + int(bool(RUN_RENDER)), label='setup')

if RUN_RENDER:
    if OVERWRITE_VIDEO and VIDEO_PATH.exists():
        VIDEO_PATH.unlink()
    render_args = [
        'python',
        f'{REPO_DIR}/scripts/render_smart_rollout_video.py',
        '--scenario-rollouts-path', str(rollout_proto_path),
        '--output-mp4', str(VIDEO_PATH),
        '--scenario-id', str(SELECTED_SCENARIO_ID),
        '--selection-strategy', str(SELECTION_STRATEGY),
        '--selection-limit', str(MAX_CANDIDATES),
        '--rollout-index', str(ROLLOUT_INDEX),
        '--scenario-tfrecords', str(SCENARIO_TFRECORDS),
        '--smart-repo-dir', str(SMART_REPO_DIR),
        '--smart-config-path', str(SMART_VAL_CONFIG),
        '--pretrain-ckpt', str(SMART_BASELINE_CKPT),
        '--processed-root', str(SMART_PROCESSED_DATA_ROOT),
        '--processed-split', 'validation',
        '--seed', '2',
        '--fps', str(FPS),
        '--dpi', str(DPI),
        '--view-radius', str(VIEW_RADIUS),
    ]
    if official_metrics_json.exists():
        render_args.extend(['--official-metrics-json', str(official_metrics_json)])
    if SCENARIO_PROTO_PATH:
        render_args.extend(['--scenario-proto-path', str(SCENARIO_PROTO_PATH)])
    if SCENARIO_PROTO_DIR:
        render_args.extend(['--scenario-proto-dir', str(SCENARIO_PROTO_DIR)])
    if FOCAL_OBJECT_ID > 0:
        render_args.extend(['--focal-object-id', str(FOCAL_OBJECT_ID)])
    if SKIP_TOKEN_TRACE:
        render_args.append('--skip-token-trace')
    render_cmd = f"cd {shlex.quote(str(REPO_DIR))} && " + ' '.join(shlex.quote(str(arg)) for arg in render_args)
    _run_cmd(render_cmd, 1 + int(bool(RUN_SETUP)), 1 + int(bool(RUN_SETUP)), label='render')
else:
    print('SMART_RUN_RENDER=0 -> skipping video rendering.')


RUN_SETUP: True
RUN_RENDER: True
video_exists: False
video_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/visualizations/smart_baseline_9e455a0332beef76_rollout0.mp4
smart_repo_sync:
{
  "mode": "cloned",
  "repo_rev": "42e6585"
}
[setup 1/2] cd /content/wosac-sim-agents-experiments && python /content/wosac-sim-agents-experiments/scripts/ensure_smart_train_runtime.py --smart-repo-dir /content/SMART
[smart-train-setup] missing pytorch_lightning: ModuleNotFoundError: No module named 'pytorch_lightning'
[smart-train-setup] $ /usr/bin/python3 -m pip install --upgrade pytorch-lightning>=2.4,<2.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.9 MB/s eta 0:00:00
[smart-train-setup] missing waymo_open_dataset: ModuleNotFoundError: No module named 'waymo_open_dataset'
[smart-train-setup] $ /usr/bin/python3 -m pip install --upgrade --no-deps waymo-open-datas

CalledProcessError: Command '['bash', '-lc', "export PYTHONUNBUFFERED=1; cd /content/wosac-sim-agents-experiments && python /content/wosac-sim-agents-experiments/scripts/render_smart_rollout_video.py --scenario-rollouts-path /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/rollout_protos/smart_baseline.binproto --output-mp4 /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260404T162859Z/visualizations/smart_baseline_9e455a0332beef76_rollout0.mp4 --scenario-id 9e455a0332beef76 --selection-strategy representative_safe --selection-limit 5 --rollout-index 0 --scenario-tfrecords 'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-*' --smart-repo-dir /content/SMART --smart-config-path experiments/smart-baseline/configs/validation_scalable_paper_repro.yaml --pretrain-ckpt /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_smoke/epoch=30.ckpt --processed-root /c

In [ ]:
# Step 5: Inspect the saved artifacts and preview the MP4.
import json
from IPython.display import Video, display

print('video_exists:', VIDEO_PATH.exists())
print('video_size:', VIDEO_PATH.stat().st_size if VIDEO_PATH.exists() else 0)
print('summary_exists:', SUMMARY_JSON_PATH.exists())
print('token_trace_exists:', TOKEN_TRACE_JSON_PATH.exists())

if SUMMARY_JSON_PATH.exists():
    payload = json.loads(SUMMARY_JSON_PATH.read_text())
    print(json.dumps(payload, indent=2, sort_keys=True))

if VIDEO_PATH.exists() and VIDEO_PATH.stat().st_size > 0:
    display(Video(str(VIDEO_PATH), embed=False))
